## Similitud - Funciones

Parámetros y funciones compartidas de MinHash, Jaccard, LSH y ventana mensual

- No se ejecuta solo: `a`, `b`, `c` y `d` lo cargan con `%run` en Utilidades

### Imports

In [ ]:
import numpy as np
from pyspark.sql import functions as F

### Constantes

In [ ]:
SEED = 42                  # semilla fija: mismos coeficientes de hash y mismas muestras en cada corrida
T = 100                    # funciones hash (largo de la firma): error de MinHash ≈ 1/(2√T) = 0.05; medido 0.027 en a_firmas_minhash
P = (1 << 31) - 1          # primo de Mersenne 2³¹−1: el hashing universal (a·x + b) mod p necesita p primo, y a·x + b < 2⁶² cabe en int64 sin overflow
B, R = 10, 10              # bandas x filas (B·R = T): umbral LSH (1/B)^(1/R) ≈ 0.79 < UMBRAL, así P(candidato | J=0.90) ≈ 0.986 y
                           # P(candidato | J=0.50) ≈ 0.01; en b_evaluacion_lsh dio recall 100% con 95% menos comparaciones que la fuerza bruta
VENTANA_DIAS = 15          # medio mes: una compra partida se convoca en días cercanos; con 15 < 28 el 2º proceso cae siempre en el mes
                           # de análisis o el siguiente, que son los únicos que carga filtro_meses
UMBRAL = 0.90              # Jaccard ≥ 0.90 sobre shingles k=5 = mismo texto salvo detalles (nº de ítem, lote, fecha)
ANIOS = [2023, 2024, 2025] # periodo completo del dataset

COLS_SIMILITUD = ["ocid", "fecha", "anio", "mes", "periodo", "entidad", "region", "proveedor", "categoria",
                  "metodo_detalle", "n_postores", "monto_final", "tramo_monto", "descripcion", "texto", "shingles"]

### MinHash

Como en lab3.1 / lab3.2

In [ ]:
def generar_coeficientes(t=T, p=P, seed=SEED):
    rng = np.random.default_rng(seed)
    return rng.integers(1, p, size=t, dtype=np.int64), rng.integers(0, p, size=t, dtype=np.int64)


def codificar_shingles(df, col="shingles", p=P):
    # shingle -> entero en [0, p) con xxhash64 (reemplaza blake2b del lab)
    return df.withColumn("codigos", F.expr(f"transform({col}, s -> pmod(xxhash64(s), {p}L))"))


def firmar(df, a_coef, b_coef, p=P):
    # firma[i] = min((a_i*x + b_i) mod p) sobre los códigos del proceso
    minimos = [F.expr(f"array_min(transform(codigos, x -> pmod({int(a)}L * x + {int(b)}L, {p}L)))")
               for a, b in zip(a_coef, b_coef)]
    return df.withColumn("firma", F.array(*minimos)).drop("codigos")

### Similitudes

In [ ]:
def jaccard_exacto(c1, c2):
    return F.size(F.array_intersect(c1, c2)) / F.size(F.array_union(c1, c2))


def minhash_sim(f1, f2, t=T):
    return F.aggregate(F.zip_with(f1, f2, lambda x, y: (x == y).cast("int")), F.lit(0), lambda acc, v: acc + v) / F.lit(t)

### LSH

Como en lab4

In [ ]:
def lsh_bandas(df_firmas, b=B, r=R, n_cubetas=None):
    # cada firma -> b filas (banda, cubeta); cubeta = hash Murmur3 de 32 bits de las r filas de la banda
    # ~4,300 millones de cubetas posibles ≫ procesos por mes (~6.4k): dos procesos comparten cubeta casi solo
    # si sus r valores son idénticos, sin colisiones accidentales que inflen los candidatos
    # n_cubetas (experimento en b_evaluacion_lsh): cubeta = hash mod n_cubetas; None = las 2³² del hash
    def cubeta(i):
        h = F.hash(F.slice("firma", i * r + 1, r))
        return F.pmod(h, F.lit(n_cubetas)) if n_cubetas else h
    bandas = F.array(*[F.struct(F.lit(i).alias("banda"), cubeta(i).alias("cubeta")) for i in range(b)])
    return (df_firmas.withColumn("bc", F.explode(bandas))
                     .select("ocid", "entidad", "fecha", "anio", "mes", F.col("bc.banda").alias("banda"), F.col("bc.cubeta").alias("cubeta")))


def orden_par(a, b):
    # proceso 1 = el más antiguo (fecha, ocid)
    return (F.col(f"{a}.fecha") < F.col(f"{b}.fecha")) | ((F.col(f"{a}.fecha") == F.col(f"{b}.fecha")) & (F.col(f"{a}.ocid") < F.col(f"{b}.ocid")))


def candidatos_lsh(df_bandas, anio, mes, ventana=VENTANA_DIAS):
    # misma cubeta en alguna banda + misma entidad + <= ventana días + proceso 1 en el mes de análisis
    a, b = df_bandas.alias("a"), df_bandas.alias("b")
    cond = ((F.col("a.entidad") == F.col("b.entidad")) &
            (F.col("a.banda") == F.col("b.banda")) & (F.col("a.cubeta") == F.col("b.cubeta")) &
            orden_par("a", "b") &
            (F.datediff(F.col("b.fecha"), F.col("a.fecha")) <= ventana) &
            (F.col("a.anio") == anio) & (F.col("a.mes") == mes))
    return a.join(b, cond).select(F.col("a.ocid").alias("ocid_1"), F.col("b.ocid").alias("ocid_2")).distinct()


def tramo(col):
    # mismos cortes que tramo_monto del KDD, para comparar el tramo de cada proceso con el de la suma
    return (F.when(col.isNull(), "0_sin_monto").when(col < 50_000, "1_<50k").when(col < 200_000, "2_50k-200k")
             .when(col < 1_000_000, "3_200k-1M").when(col < 5_000_000, "4_1M-5M").otherwise("5_>5M"))


def verificar_pares(cands, df_procesos, df_firmas, umbral=UMBRAL):
    # Jaccard exacto sobre shingles; se conserva el par si >= umbral
    base = df_procesos.join(df_firmas.select("ocid", "firma"), "ocid")
    p1 = base.select(*[F.col(c).alias(f"{c}_1") for c in base.columns])
    p2 = base.select(*[F.col(c).alias(f"{c}_2") for c in base.columns])
    return (cands.join(p1, "ocid_1").join(p2, "ocid_2")
                 .withColumn("jaccard", jaccard_exacto(F.col("shingles_1"), F.col("shingles_2")))
                 .withColumn("minhash_sim", minhash_sim(F.col("firma_1"), F.col("firma_2")))
                 .where(F.col("jaccard") >= umbral)
                 .select("ocid_1", "ocid_2",
                         F.col("entidad_1").alias("entidad"), F.col("region_1").alias("region"), F.col("periodo_1").alias("periodo"),
                         "fecha_1", "fecha_2", F.datediff("fecha_2", "fecha_1").alias("dias_diferencia"),
                         F.round("jaccard", 4).alias("jaccard"), F.round("minhash_sim", 4).alias("minhash_sim"),
                         "monto_final_1", "monto_final_2", (F.col("monto_final_1") + F.col("monto_final_2")).alias("monto_suma"),
                         F.col("tramo_monto_1").alias("tramo_1"), F.col("tramo_monto_2").alias("tramo_2"),
                         tramo(F.col("monto_final_1") + F.col("monto_final_2")).alias("tramo_suma"),
                         F.col("metodo_detalle_1").alias("metodo_1"), F.col("metodo_detalle_2").alias("metodo_2"),
                         F.col("categoria_1").alias("categoria"), "proveedor_1", "proveedor_2", "n_postores_1", "n_postores_2",
                         (F.col("texto_1") == F.col("texto_2")).alias("texto_identico"),
                         (F.to_date("fecha_1") == F.to_date("fecha_2")).alias("mismo_dia"),
                         "descripcion_1", "descripcion_2"))

### Ventana mensual

In [ ]:
def mes_siguiente(anio, mes):
    return (anio + 1, 1) if mes == 12 else (anio, mes + 1)


def filtro_meses(anio, mes):
    # mes de análisis + mes siguiente (contexto posterior de la ventana); Spark lee sólo esas 2 carpetas
    a2, m2 = mes_siguiente(anio, mes)
    return ((F.col("anio") == anio) & (F.col("mes") == mes)) | ((F.col("anio") == a2) & (F.col("mes") == m2))